# Module 3 — Python Basics Assessment (Notebook Version)

> **Welding Image Analysis** — small images of welds taken on coupons
>
> **Time:** about 2 hours · **Turn in:** this finished notebook, `session_log.txt`, and `outputs/weld_class_highlight.png`.

## How to use this notebook

- Each **Task** has a short explanation cell followed by a code cell.
- Look for `# TODO` lines inside the code cells — those are the only places you have to write Python.
- Run cells in order from top to bottom (Shift+Enter).
- If a cell errors out, fix it before moving on — later cells build on earlier ones.

## What you'll be working with

Each welding photo contains:

- **workpieces** — the metal plates where the welding line is applied
- **welding lines** — the bead the torch lays down
- **backgrounds** — the area not occupied by the workpiece or the welding lines.

You won't train a neural network — you'll use plain Python (variables, lists, dictionaries, loops, `if`/`else`) to **load**, **count**, and **highlight** what's in those images. By the end you'll classify every pixel in an image according to simple rules that you define.

You do **not** need to know image processing — six helper functions in `utils/welding_utils.py` do the loading and plotting for you.

## Setup (run me first)

This cell:

1. Adds the project root to `sys.path` so Python can find `utils/welding_utils.py`.
2. Imports the helper functions you'll use throughout the assessment.
3. Sets up file paths to the dataset and the log file.
4. Tells matplotlib to draw figures **inline** (so plots show up here in the notebook).

You should not need to change anything in this cell.

In [ ]:
%matplotlib inline

import os
import sys

# Find the project root (one level up from this notebook).
ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) \
    if os.path.basename(os.getcwd()) == "notebooks" \
    else os.getcwd()
sys.path.insert(0, ROOT)

from utils.welding_utils import (
    load_coco_metadata,
    load_image_small,
    plot_channel,
    plot_image,
    plot_list,
    plot_overlay,
    save_current_figure
)

DATA_DIR   = os.path.join(ROOT, "data", "Welding Defect Detection.v1i.coco", "test")
META_PATH  = os.path.join(DATA_DIR, "_annotations.coco.json")
LOG_PATH   = os.path.join(ROOT, "session_log.txt")
OUTPUT_DIR = os.path.join(ROOT, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project root:", ROOT)
print("Dataset dir :", DATA_DIR)
print("Log file    :", LOG_PATH)

### Logging helper

Throughout the assessment we'll keep a running log of what we do — like a quality engineer fills out a checklist on the shop floor. The two helpers below:

- `log("...")` — print a line **and** write it to `session_log.txt`.
- `reset_log()` — empty the log at the start of a fresh run.

You don't need to write these — just run the cell so they're available.

In [ ]:
def log(message):
    print(message)
    with open(LOG_PATH, "a") as f:
        f.write(message + "\n")


def reset_log():
    open(LOG_PATH, "w").close()


reset_log()
log("Notebook session started.")

## Task 1 — Write your name to the log

The very first entry in the log file should be your name.

In [ ]:
# TODO: Change the value of the variable 'name' with your actual name.
name = None
if name is None:
    raise ValueError("Name must have a value.")
log(f"Student: {name}")

### Load the dataset metadata

This loads the COCO annotation file as a Python dictionary. We'll reuse it in Tasks 7, 8, and 11.

In [ ]:
metadata = load_coco_metadata(META_PATH)
log(f"Loaded {len(metadata['images'])} test images.")

## Task 2 — Make one of every kind of variable

In welding inspection you have to track many *kinds* of information at once. Create one variable of each basic type. Suggested values are in the comments — feel free to change them.

Then **reassign** each one to a different value of the same type.

In [ ]:
log("\n=== Task 2: variables ===")

# TODO: replace each ellipsis (...) with an example value of the right data type.
# For example, if the variable is supposed to be a string, you could assign it a value like "VF9 CNC".
operator_id   = ...   # int
torch_voltage = ...   # float
station_name  = ...   # str
is_calibrated = ...   # bool
shift_codes   = ...   # list
torch_xyz     = ...   # tuple
weld_record   = ...   # dict with keys like "part_id", "pass_number", and "timestamp"

# assert that each variable is of the correct type
assert isinstance(operator_id, int), "operator_id should be an int"
assert isinstance(torch_voltage, float), "torch_voltage should be a float"
assert isinstance(station_name, str), "station_name should be a str"
assert isinstance(is_calibrated, bool), "is_calibrated should be a bool"
assert isinstance(shift_codes, list), "shift_codes should be a list"
assert isinstance(torch_xyz, tuple), "torch_xyz should be a tuple"
assert isinstance(weld_record, dict), "weld_record should be a dict"

# TODO: now reassign each one to a NEW value of the same type:
operator_id   = ...
torch_voltage = ...
station_name  = ...
is_calibrated = ...
shift_codes   = ...
torch_xyz     = ...
weld_record   = ...

log(f"operator_id={operator_id}, torch_voltage={torch_voltage}, "
    f"station_name={station_name}, is_calibrated={is_calibrated}")
log(f"shift_codes={shift_codes}, torch_xyz={torch_xyz}, weld_record={weld_record}")

## Task 3 — Variable name rules

There are strict rules on what a variable can be named. For all of the variable names below, replace the **?** with  **OK** or **NOT OK** and add a quick reason.

Hints: variable names cannot start with a digit, cannot contain hyphens, and cannot be a Python keyword like `class`.

Type your answers in the cell below (it's a markdown cell — double-click to edit, then Shift+Enter to render).

**Your answers (edit this cell):**

- `weld_pass_2`        → ?
- `2nd_weld_pass`      → ?
- `weldPass`           → ?
- `weld-pass`          → ?
- `class`              → ?
- `WELD_LIMIT_VOLTS`   → ?

## Task 4 — A function that returns two numbers

Finish the function below that returns the result of a division operation on integers.


Make use of the `//` (integer divide) and `%` (remainder) operators.

In [ ]:
def divmod_pair(a, b):
    quotient  = ...   # TODO 
    remainder = ...   # TODO 
    return (quotient, remainder)


assert divmod_pair(10, 3) == (3, 1), "divmod_pair(10, 3) should return (3, 1)"

log("\n=== Task 4: divmod_pair ===")
q, r = divmod_pair(217, 8)
log(f"217 weld passes / 8 per shift = {q} shifts, {r} leftover passes")

## Task 5 — Check the type of a value

This function (already written for you) prints a different message based on the type of `value`. Just **call it** on at least 5 different values from Task 2 in the second cell below.

> **Note:** the function checks `bool` *before* `int` because in Python `True` is technically also an `int`.

In [ ]:
def report_value(value):
    if isinstance(value, bool):
        log(f"  Boolean value: {value}")
    elif isinstance(value, int):
        log(f"  Integer value: {value}")
    elif isinstance(value, float):
        log(f"  Float value: {value}")
    elif isinstance(value, str):
        log(f"  String value: '{value}'")
    elif isinstance(value, list):
        log(f"  List of {len(value)} items")
    else:
        log(f"  Other type: {type(value).__name__}")

In [ ]:
log("\n=== Task 5: report_value ===")

# TODO: call report_value() on at least 5 different values from Task 2.


## Task 6 — Strings: compare, concatenate, f-string

### 6a. Compare two part numbers, ignoring case and spaces
In this task you will write a function named `same_part` that takes in two strings and return `True` if they are equal, or `False` if they aren't.

You will want to make use of the functions strip() and lower(). We have already used lower in class--strip is another string method that removes whitespace (or any specified characters) from the beginning and end of a string. This is crucial for guaranteeing that any leading or trailing characters are not interfering with your string comparisons.


There are many places online to search for documentation to learn more about built in methods and types. The official python docs are at docs.python.org, but many other resources can be just as helpful.

In [ ]:
# TODO: write a function named same_part to compare two part IDs.


log("\n=== Task 6: strings ===")
log(f"same_part('  WP-001 ', 'wp-001') = {same_part('  WP-001 ', 'wp-001')}")

### 6b. Build the same string two different ways

In this task you will build the same string using two different methods: the concatenation operator (`+`) and f-strings. 

The string should structure the station and voltage variables in the following format:

`Station Cell-04 | V=22.5`

If you need a refresher on how f-strings work, do a quick web search on them.

In [ ]:
station = "Cell-04"
voltage = 22.5

line_a = ... # TODO build the string using the concatenation operator (+) and the variables station and voltage.
line_b = ... # TODO build the string using an f-string and the variables station and voltage.

assert line_a == line_b == "Station Cell-04 | V=22.5", "The two lines should be the same"

log(line_a)
log(line_b)

## Task 7 — Lists

You'll work with the **list of image file names** from the dataset. These filenames are **strings** that are stored within the nested structure of the JSON file (located at `data\Welding Defect Detection.v1i.coco\test\_annotations.coco.json`). JSON files are organized like a python `dict` structure. You should look through the file for a key named **"file_name"**.

The JSON file has already been loaded for you by a previous cell. It is stored as a `dict` in the variable named `metadata`.

Your task is to pull all of the filenames out of the metadata dictionary and enter them into the `file_names` list variable (which we have already defined for you). You will need to open the metadata file and examine it to figure out which keys need to be accessed to obtain this information.

Hint: there is a nested list that contains all of the metadata (in dictionary form) for every image. You will want to iterate through this list with a for in statement, and then append the filename record in the variables `file_names`.

In [ ]:
log("\n=== Task 7: lists ===")

file_names = []
# TODO access the 'images' list from the metadata variable (a dictionary)
# TODO Use a for in loop to access all of the filenames from the 'images' list and append them to the file_names list

log(f"Built file_names with {len(file_names)} entries.")

### 7a. Index and slice
Without using the len() function, use slicing to extract sublists of the file_names list, per the instructions below.

In [ ]:
log(f"first  : {file_names[0]}")

last = ...          # TODO: obtain the last item using negative indexing
first_five = ...    # TODO: obtain the first five items
last_three = ...    # TODO: obtain the last three items.

log(f"last   : {last}")
log(f"first 5: {first_five}")
log(f"last 3 : {last_three}")

### 7b. Edit one item
Change the file name of the 3rd file in the list to "RENAMED.jpg"

In [ ]:
third_file = ... # TODO access the third file
before_edit = str(third_file)

... # TODO change the name of the third file to "RENAMED.jpb"

after_edit = ... # TODO access the third file again

assert after_edit != before_edit, "The filename should have changed after the edit."

log(f"before edit: {before_edit}\n, after edit: {after_edit}")

### 7c. Delete items

Using the del operator, remove the 5th and 7th items in the list. 

In [ ]:
len_before = len(file_names)
file5 = file_names[4]
file7 = file_names[6]

... # TODO delete the 5th and 7th items in the list.

assert file5 != file_names[4]
assert file7 != file_names[6]

log(f"after delete: {len(file_names)} files left")

### 7d. The `+` and `*` operators on lists
Complete the code according to the instructions on each line with the TODO label

In [ ]:
header      = ["frame_index", "file_name"]
defect_cols = ["n_defects", "n_welds"]
csv_header  = ... # TODO use the string concatenation operator to join the two lists into one
empty_row   = ... # TODO Create a list of zeros with the same length as csv_header (hint: use the * operator to repeat the integer 0)

assert len(csv_header) == len(header) + len(defect_cols), "csv_header should contain all the columns from header and defect_cols"
assert len(empty_row) == len(csv_header), "empty_row should have the same number of items as csv_header"

log(f"csv_header={csv_header}")
log(f"empty_row ={empty_row}")

### 7e. Unpack a list

The dataset stores defect bounding boxes that enclose the weld as 4-element lists `[x, y, w, h]`. Unpack the first one in the list into four variables: `x, y, w, h`.

Hint: there is a key called "annotations". The bounding boxes will be contained within the key 'bbox'.

In [ ]:
first_box = ... # TODO access the first bounding box from the metadata
# TODO unpack the bounding box into its components, assigning them to variables named x, y, w, and h.

assert x == 153 and y == 0 and w == 349.5 and h == 246.5, "Bounding box values do not match expected values."

log(f"first defect box: x={x}, y={y}, w={w}, h={h}")

### 7f. Look up an item, then insert and append

Both `insert` and `append` make the list one longer. Use these functions to add an item to the beginning and end of the file_names list

In [ ]:
first_filename = file_names[0] 
log(f"index of first file = {file_names.index(first_filename)}") # checking index of first filename
length_before = len(file_names)

... # TODO insert "calibration_frame.jpg" at the beginning of the list
... # TODO insert "end_of_run.jpg" at the end of the list

assert length_before + 2 == len(file_names), "Two file names should have been added to the list."

log(f"file_names now has {len(file_names)} items")

## Task 8 — Tuples and dictionaries

### 8a. Make a tuple from a slice of a list

In [ ]:
log("\n=== Task 8: tuples and dicts ===")

defect_classes = ["Defect", "Welding Line", "Workpiece"]
class_tuple = ... # TODO turn the first two entries in defect_classes into a tuple 

assert isinstance(class_tuple, tuple), "class_tuple should be a tuple"
assert class_tuple == ("Defect", "Welding Line"), "class_tuple should contain the first two defect classes"

log(f"class_tuple = {class_tuple}")

### 8b. Add a `length` key to the dataset

In [ ]:
... # TODO create a new entry in the metadata dict named "length", equal to the number of file_names

log(f"metadata['length'] = {metadata['length']}")

## Task 9 — Image functions

We'll work on a 64×64 version of the first welding image so loops finish in well under a second.

In [ ]:
log("\n=== Task 9: image functions ===")

first_file = metadata["images"][0]["file_name"]
pixels     = load_image_small(os.path.join(DATA_DIR, first_file))
log(f"loaded image as {len(pixels)} rows of {len(pixels[0])} pixels")

# Take a quick look at the original image first.
plot_image(pixels, title="The image you'll be analyzing")

### 9a. RGB → brightness

Use the standard formula: `0.299 * R + 0.587 * G + 0.114 * B`.

Quick check after you write it: `rgb_to_intensity((255, 0, 0))` should be about `76.245`.

In [ ]:
def rgb_to_intensity(rgb):
    # TODO unpack the rgb tuple into its r, g, and b components
    brightness = ... # TODO compute the brightness intensity using the formula above
    return ... # TODO return the brightness value rounded to three decimal places
    

assert rgb_to_intensity((255, 0, 0)) == 76.245, "Intensity of pure red should be 76.245"
assert rgb_to_intensity((0, 255, 0)) == 149.685, "Intensity of pure green should be 149.685"
assert rgb_to_intensity((0, 0, 255)) == 29.07, "Intensity of pure blue should be 29.07"

log(f"rgb_to_intensity((255,0,0)) = {rgb_to_intensity((255, 0, 0))}")

### 9b. Average brightness of a list of pixels
To compute the average value of a list:

1. Start a running total at `0`.
2. Loop through each item and add it to the total.
3. Divide the total by the number of items in the list.

Formula:

`average = sum_of_values / number_of_values`

For pixel brightness, this means:
- convert each RGB pixel to intensity,
- add all intensities,
- divide by how many pixels are in the list.

In [ ]:
def average_intensity(rgb_list):
    total = 0.0 # sum of intensities of every pixel
    # TODO Loop through each RGB pixel in the list, adding their intensities to the 'total' variable

    return -1


sample_pixels = [(0, 0, 0), (255, 255, 255), (255, 0, 0)]
assert round(average_intensity(sample_pixels), 3) == 110.415, "Average intensity test failed"

### 9c. Min and max 
This task is similar to the prior one, except that you are using builtin python functions to return a tuple of variables (the minimum and maximum intensities) instead of a single average intensity. 

Do a web search to discover the builtin functions that will compute the minimum and maximum of a list.

Hint: To simplify your approach, create a new variable to store the intensities of every pixel. Then iterate across the pixels, appending their intensities to the list as you go along.



In [ ]:
def min_max_intensity(rgb_list):
    # TODO: return a tuple with the minimum and maximum intensity values.
    return ...


test_min_max = min_max_intensity([(0, 0, 0), (255, 255, 255), (255, 0, 0)])
assert test_min_max == (0.0, 255.0), "min_max_intensity should return (0.0, 255.0) for the test input"

### 9d. Stats for every row


In [ ]:
def row_stats(pixels):
    # List to store results for each row
    results = []

    # TODO: loop over each row in pixels, collecting information 
    # about the row length, average intensity, and min/max intensity for each row.

    return results


test_image = [
    [(0, 0, 0), (255, 255, 255)],
    [(255, 0, 0), (0, 255, 0)],
]

stats = row_stats(test_image)
assert len(stats) == 2, "row_stats should return one tuple per row"
assert stats[0][0] == 2, "First row length should be 2"
assert round(stats[0][1], 3) == 127.5, "First row average should be 127.5"
assert round(stats[1][1], 3) == 112.965, "Second row average should be 112.965"

### 9e. Flatten an image and collect statistics on all pixels
Flattening an image involves transforming it from a multi-dimensional array into a single array. Typically, this retains the top-down, left-right ordering of the pixels (as if you were reading them like a book). 

By flattening the image, we can feed it's entire contents as a single list into the functions you just created.

Create a function named flatten to convert each image into a single 1D list.

Hint: You will want to initialize an empty list, iterate across every pixel row, and append pixels to your empty list.

In [ ]:
# TODO create the flatten function that receives a single parameter, 'pixels'

# test your flatten function
flat = flatten(pixels)
assert len(flat) == len(pixels) * len(pixels[0]), "The flattened list should have as many pixels as the original image"

log(f"average brightness of whole image = {round(average_intensity(flat), 2)}")
mn, mx = min_max_intensity(flat)
log(f"min/max brightness = {round(mn, 2)} / {round(mx, 2)}")

delta = average_intensity(pixels[0]) - average_intensity(pixels[-1])
log(f"|top-bottom| brightness = {round(abs(delta), 2)} (type={type(delta).__name__})")

### 9f. Plot a channel and a brightness curve

We have created several helper functions to plot lists of pixels. Each `plot_*` call shows a figure inline in your notebook. 

Two example plots are shown below, where the red channel is plotted, as well as the intensity of each pixel (resulting in a grayscale image).

The plot_list function plots a list as the y-value on a line graph. Extract the row statistics from the pixels variable using row_stats. Then, create a list called row_brightness that contains the average intensity parameter for each row from your stats variable. It will be plotted on the graph below.

Hint: Call row_stats on the pixels variable, then loop through the stats list, and access the tuple index that contains the average brightness parameter. Then append it to the list.


In [ ]:
plot_channel(pixels, channel="R",         title="Red channel")
plot_channel(pixels, channel="intensity", title="Brightness")

# TODO get the row statistics 
# TODO extract the brightnesses from the row statistics into a list named row_brightness

plot_list(row_brightness, title="Brightness per row")

## Task 10 — Threshold pixels and highlight the weld bead

### 10a. Classify every pixel with `if` / `elif` / `else`

Thresholding is a simple technique where every pixel or feature of an image is compared to some value to determine what category it belongs to. For example, you might try to detect the location of apples on a converyor belt by applying the following rule:

- if the pixel contains a red value greater than 180 AND green and blue values less than 120, the pixel belongs to an apple
- else the pixel belongs to the background

For this task, we are going to look at every RGB pixel in the image and classify them according to a set of rules defined by if-elif-else statements. The looping structure has been created for you--your goal is to create thresholding rules that, as neatly as possible, classify a region of the image as belonging to the 'weld', 'workpiece', or 'background'. Your rules will be applied to the image and plotted to the notebook.

In [ ]:
log("\n=== Task 10: threshold + highlight ===")

label_dict = {
    0 : "weld",
    1: "workpiece",
    2: "background"
}
color_mapping = {
    0: "red",    # red for weld
    1: "green",    # green for workpiece
    2: "blue",    # blue for background
}

def classify_pixel(rgb):
    # TODO write your custom thresholding rules here.
    return -1

def classify_pixels(pixels):
    h = len(pixels)
    w = len(pixels[0])
    labels = [] # this will be a 2D list of the same shape as pixels
    for y in range(h):
        row_labels = []
        for x in range(w):
            row_labels.append(classify_pixel(pixels[y][x]))
        labels.append(row_labels)
    return labels

# plot the image labels
labels = classify_pixels(pixels)

def plot_label_overlay(pixels, labels, label_dict=label_dict):
    label_colors = {
        0: [255, 0, 0],    # red for weld
        1: [0, 255, 0],    # green for workpiece
        2: [0, 0, 255],    # blue for background
    }
    h = len(pixels)
    w = len(pixels[0])
    overlay = [[None for _ in range(w)] for _ in range(h)]
    for y in range(h):
        for x in range(w):
            label = labels[y][x]
            overlay[y][x] = label_colors.get(label, [0, 0, 0])
    import matplotlib.pyplot as plt
    # plot the overlaid image
    plt.figure(figsize=(8, 8))
    plt.imshow(pixels)
    plt.imshow(overlay, alpha=0.5)
    plt.title("Label overlay")
    plt.axis("off")
    # save the figure to the outputs directory
    save_current_figure(os.path.join(OUTPUT_DIR, "weld_class_highlight.png"))
    plt.show()



plot_label_overlay(pixels, labels, label_dict)
for i in range(3):
    print(f"{label_dict[i]} == {color_mapping[i]}", end="; ")